In [2]:
import json
import sys
import argparse

sys.path.insert(0, ".")

import torch
torch.set_grad_enabled(False)

# Notebook-style args as a dict:
from types import SimpleNamespace

args = SimpleNamespace(
    model_id="google/gemma-3-4b-it",
    sae_release="gemma-scope-2-4b-pt-res",
    sae_id="layer_29_width_65k_l0_medium",
    layer=29,
    n_sentences=100,
    data_path="data/flores_5lang_N1000.json"
)
LANGUAGES = ["French", "English", "German", "Dutch", "Italian"]


In [3]:
!ls

CLAUDE.md  data  lib  notebooks  references  scripts


In [4]:
from lib.core import *

# Loading SAE, Model and Data

In [5]:
print("Loading model + SAE...")
model, tokenizer, sae = load_model_and_sae(args.model_id, args.sae_release, args.sae_id)

with open(args.data_path) as f:
    flores = json.load(f)[:args.n_sentences]

Loading model + SAE...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [6]:
print(f"FINDING LANGUAGE FEATURES | layer={args.layer} | {args.n_sentences} sentences")


FINDING LANGUAGE FEATURES | layer=29 | 100 sentences


In [7]:
mean_acts = {}
for lang in LANGUAGES:
    texts = [entry[lang] for entry in flores]
    mean_acts[lang] = get_mean_sae_activations(model, tokenizer, sae, args.layer, texts)
    print(f"  {lang}: {(mean_acts[lang] > 0).sum().item()} active features")

  French: 20446 active features
  English: 19479 active features
  German: 20391 active features
  Dutch: 20679 active features
  Italian: 20998 active features


In [8]:
lang_features = {}
for lang in LANGUAGES:
    target = mean_acts[lang]
    other_max = torch.stack([mean_acts[l] for l in LANGUAGES if l != lang]).max(dim=0).values
    spec = target - other_max
    spec[target <= 0] = -float("inf")
    top_vals, top_idxs = spec.topk(10)

    print(f"\n  {lang}:")
    clean = []
    for idx, val in zip(top_idxs, top_vals):
        i = idx.item()
        ratio = target[i].item() / max(other_max[i].item(), 0.01)
        tag = " ***" if ratio > 100 else " **" if ratio > 10 else ""
        print(f"    {i:>8} | act={target[i]:>7.1f} | others={other_max[i]:>6.1f} | {ratio:>7.0f}x{tag}")
        if ratio > 3.0:
            clean.append(i)
    lang_features[lang] = clean


  French:
         205 | act= 2678.7 | others=   1.7 |    1589x ***
        1387 | act= 1176.1 | others=   0.3 |    3849x ***
        9269 | act=  400.7 | others=   1.0 |     400x ***
        2265 | act=  369.6 | others=   0.2 |    2227x ***
       17376 | act=  322.3 | others=   0.0 |   32233x ***
        6150 | act=  321.5 | others=   1.9 |     172x ***
        6389 | act=  247.4 | others=   2.7 |      91x **
       12096 | act=  227.2 | others=   1.0 |     217x ***
       13050 | act=  214.8 | others=   0.2 |    1077x ***
       30363 | act=  142.7 | others=   1.0 |     139x ***

  English:
        1832 | act=  381.5 | others= 162.6 |       2x
        4158 | act=  367.8 | others= 168.9 |       2x
        3510 | act=  307.0 | others= 138.4 |       2x
         803 | act=  312.6 | others= 174.5 |       2x
        1491 | act=  583.5 | others= 446.6 |       1x
         737 | act=  550.1 | others= 420.2 |       1x
        1972 | act=  447.4 | others= 342.1 |       1x
        1999 | act= 

In [9]:
fr_feats = lang_features["French"][:1]
fr_text = flores[min(10, len(flores) - 1)]["French"][:150]
fr_text_2 = flores[min(20, len(flores) - 1)]["French"][:150]
print(f"\n{'='*70}")
print(f"SUPPRESSION | French features={fr_feats}")
print(f"{'='*70}")

suppress_prompts = [
    ("PRODUCE French", "Write a short paragraph in French about the weather."),
    ("COMPREHEND French", f"Translate to English: '{fr_text}'"),
    ("COMPREHEND French", f"Translate to English: '{fr_text_2}'"),
    ("CONTROL German", "Please say a sentence in German."),
    ("CONTROL English", "What is the capital of France?"),
]

def colorize(text, color):
    code = {"green": "\033[92m", "red": "\033[91m", "end": "\033[0m"}
    return f"{code[color]}{text}{code['end']}"

for label, prompt in suppress_prompts:
    baseline = generate(model, tokenizer, prompt, layer=args.layer)
    print(f"\n  [{label}]")
    print(f"    baseline: {colorize(baseline[:140], 'green')}")
    for s in [3, 6]:
        hook = make_suppression_hook(sae, fr_feats, strength=s)
        out = generate(model, tokenizer, prompt, hook_fn=hook, layer=args.layer)
        print(f"    s={s}:      {colorize(out[:140], 'red')}")

The following generation flags are not valid and may be ignored: ['top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



SUPPRESSION | French features=[205]

  [PRODUCE French]
    baseline: Okay, here's a short paragraph in French about the weather:

"Aujourd'hui, il fait un temps magnifique ! Le soleil brille et le ciel est ble
    s=3:      Okay, here's a short paragraph in French about the weather:

"Today, the weather is quite pleasant. The sun is shining brightly, and there's
    s=6:      Okay, here's a short paragraph in French about the weather: today, the weather is quite pleasant. inherently, the sun is shining brightly, a

  [COMPREHEND French]
    baseline: Here are a few options for translating that sentence, with slightly different nuances:

**Option 1 (Most straightforward):**

“At 11:29, the
    s=3:      Here are a few options for translating the French sentence, with slightly different nuances:

**Option 1 (Most straightforward):**

“At 11:2
    s=6:      Here are a few options for translating the sentence, with slightly different nuances:

**Option 1 (Most straightforward):**

“At 11

# Scaling (Clamping French Features)

In [11]:
top_feat = lang_features["French"][0] if lang_features["French"] else None
print(f"\n{'='*70}")
print(f"STEERING | French feature={top_feat} | norm-scaled, KV-cache aware")
print(f"{'='*70}")

steer_prompts = [
    ("Capital", "What is the capital of Germany?"),
    ("Science", "Explain briefly why the sky is blue."),
    ("Fun fact", "Tell me a fun fact."),
]

def colorize(text, color):
    code = {"green": "\033[92m", "red": "\033[91m", "end": "\033[0m"}
    return f"{code[color]}{text}{code['end']}"

if top_feat is not None:
    for label, prompt in steer_prompts:
        baseline = generate(model, tokenizer, prompt, layer=args.layer)
        print(f"\n  [{label}]")
        print(f"    baseline:   {colorize(baseline[:140], 'green')}")
        for c in [0.05, 0.1, 0.2]:
            hook = make_steering_hook(sae, top_feat, coeff=c)
            out = generate(model, tokenizer, prompt, hook_fn=hook, layer=args.layer)
            print(f"    coeff={c}: {colorize(out[:140], 'red')}")

    # Also try steering 8 layers before SAE (Google's approach — stronger effect)
    steer_early = max(0, args.layer - 8)
    print(f"\n  (steering at layer {steer_early} — 8 layers before SAE, stronger effect)")
    for label, prompt in steer_prompts:
        for c in [0.05, 0.1]:
            hook = make_steering_hook(sae, top_feat, coeff=c)
            out = generate(model, tokenizer, prompt, hook_fn=hook, layer=steer_early)
            print(f"    [{label}] coeff={c} L{steer_early}: {colorize(out[:140], 'red')}")
else:
    print("  No clean French features found, skipping steering.")

print(f"\n{'='*70}")
print("DONE")
print(f"{'='*70}")



STEERING | French feature=205 | norm-scaled, KV-cache aware

  [Capital]
    baseline:   The capital of Germany is **Berlin**. 

It’s a vibrant and historically significant city! 😊 

Do you want to know anything more about Berlin
    coeff=0.05: The capital of Germany is **Berlin**. 

It’s a vibrant, historic, and important city! 😊 

Do you want to know anything else about Berlin, or
    coeff=0.1: The capital de Allemagne est **Berlin**. 

(The capital de Allemagne est Berlin.) 

Berlin est une ville avec une histoire riche et un impor
    coeff=0.2: Le Berlin est la capitale de l'Allemagne. 

(Berlin est la capitale de l'Allemagne.)

  [Science]
    baseline:   The sky is blue due to a phenomenon called **Rayleigh scattering**. Here's the breakdown:

* **Sunlight is made of all colors:** White sunli
    coeff=0.05: The sky is blue due to a phenomenon called **Rayleigh scattering**. 

Here's the simple version:

* **Sunlight is made of all colors.**
* **
    coeff=0.1: The sky est bl

In [13]:
!pip install fasttext

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached pybind11-3.0.2-py3-none-any.whl.metadata (10 kB)
Using cached pybind11-3.0.2-py3-none-any.whl (310 kB)
  Created wheel for fasttext: filename=fasttext-0.9.3-cp312-cp312-linux_x86_64.whl size=338182 sha256=22e44756b1203f159f3417667ddc6eb6de888cad29fd5371ee252b7cc50cab8e
  Stored in directory: /home/snfiel01/.cache/pip/wheels/20/27/95/a7baf1b435f1cbde017cabdf1e9688526d2b0e929255a359c6
Successfully built fasttext
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [fasttext]
